# Consultas y bibliotecas

Consultas 1 a 17 sobre la base de datos de elementos transponibles (TEs) en MongoDB.

In [4]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))  # para poder hacer "from pipeline.limpiar import clave"

import pandas as pd
from IPython.display import display
from pymongo import MongoClient

# Conexión a la base ya cargada (NO usar setup_database con drop_existing=True: borraría las colecciones)
MONGO_URI = "mongodb://localhost:27017"  # ajusta si tu Mongo no es local
NOMBRE_DB = "panteon"

client = MongoClient(MONGO_URI)
db = client[NOMBRE_DB]

# Verificación rápida
print(db.list_collection_names())
print("sequences:", db.sequences.count_documents({}), "| species:", db.species.count_documents({}))

['sequences', 'species']
sequences: 239946 | species: 2790


## Bloque 1: Cantidad de elementos (Consultas 1 a 4)

### Consulta 1
¿Cuántos TEs hay por clase, por orden y por superfamilia en toda la base?

In [5]:
# 1A. Conteo por Clase de TE
pipeline_q1_clase = [
    {"$group": {"_id": "$te_class", "total_tes": {"$sum": 1}}},
    {"$sort": {"total_tes": -1}}
]
df_q1_clase = pd.DataFrame(list(db.sequences.aggregate(pipeline_q1_clase)))
display(df_q1_clase)

# 1B. Conteo por Orden de TE
pipeline_q1_orden = [
    {"$group": {"_id": "$te_order", "total_tes": {"$sum": 1}}},
    {"$sort": {"total_tes": -1}}
]
df_q1_orden = pd.DataFrame(list(db.sequences.aggregate(pipeline_q1_orden)))
display(df_q1_orden)

# 1C. Conteo por Superfamilia de TE (aquí sí usamos json_normalize porque _id es un diccionario)
pipeline_q1_superfamilia = [
    {"$group": {
        "_id": {
            "clase": "$te_class",
            "orden": "$te_order",
            "superfamilia": "$te_superfamily"
        },
        "total_tes": {"$sum": 1}
    }},
    {"$sort": {"total_tes": -1}}
]
df_q1_super = pd.json_normalize(list(db.sequences.aggregate(pipeline_q1_superfamilia)))
display(df_q1_super.head(15))

,_id,total_tes
0,CLASSI,194774
1,CLASSII,45172


,_id,total_tes
0,LTR,119368
1,LINE,68382
2,TIR,36471
3,HELITRON,5075
4,SINE,3634
5,MITE,3456
6,DIRS,2933
7,PLE,344
8,RETROPOSON,113
9,CRYPTON,112


,total_tes,_id.clase,_id.orden,_id.superfamilia
0,50849,CLASSI,LTR,COPIA
1,43963,CLASSI,LTR,GYPSY
2,23821,CLASSI,LINE,L1
3,12202,CLASSI,LTR,ERV
4,11234,CLASSI,LINE,CR1
5,10962,CLASSI,LINE,L2
6,10729,CLASSII,TIR,TC1MARINER
7,10648,CLASSII,TIR,HAT
8,5983,CLASSII,TIR,TIR
9,5143,CLASSI,LINE,RTE


### Consulta 2
¿Cuántos TEs tiene cada especie, desglosados por clase, orden y superfamilia?

In [3]:
pipeline_q2 = [
    {"$group": {
        "_id": {
            "species_key": "$species_key",
            "te_class": "$te_class",
            "te_order": "$te_order",
            "te_superfamily": "$te_superfamily"
        },
        "cantidad_tes": {"$sum": 1}
    }},
    {"$sort": {"_id.species_key": 1, "cantidad_tes": -1}}
]

res_q2 = list(db.sequences.aggregate(pipeline_q2))
# Aplanamos el diccionario _id para verlo limpio en un DataFrame
df_q2 = pd.json_normalize(res_q2)
display(df_q2.head(15))

,cantidad_tes,_id.species_key,_id.te_class,_id.te_order,_id.te_superfamily
0,1,abortiporus biennis,CLASSI,LTR,BELPAO
1,58,abrus pulchellus subsp cantoniensis,CLASSI,LTR,COPIA
2,58,abrus pulchellus subsp cantoniensis,CLASSI,LTR,GYPSY
3,18,abrus pulchellus subsp cantoniensis,CLASSI,LTR,CAULIMOVIRUS
4,14,abrus pulchellus subsp cantoniensis,CLASSII,TIR,HAT
5,8,abrus pulchellus subsp cantoniensis,CLASSII,TIR,CMC
6,7,abrus pulchellus subsp cantoniensis,CLASSII,TIR,MULE
7,7,abrus pulchellus subsp cantoniensis,CLASSI,LINE,L1
8,3,abrus pulchellus subsp cantoniensis,CLASSI,LTR,LARD
9,3,abrus pulchellus subsp cantoniensis,CLASSII,HELITRON,HELITRON


### Consulta 3
¿Cómo cambia la cantidad de TEs por clase y orden entre reinos (Animalia, Plantae, Fungi)? ¿Y entre phyla?

In [6]:
# 3A. Cantidad de TEs por Clase y Orden entre Reinos (Kingdom)
pipeline_q3_reino = [
    # Paso 1: Pre-agrupar en sequences por especie, clase y orden para reducir volumen
    {"$group": {
        "_id": {
            "species_key": "$species_key",
            "te_class": "$te_class",
            "te_order": "$te_order"
        },
        "conteo_especie": {"$sum": 1}
    }},
    # Paso 2: Cruzar con la colección species usando el índice único species_key
    {"$lookup": {
        "from": "species",
        "localField": "_id.species_key",
        "foreignField": "species_key",
        "as": "sp_info"
    }},
    {"$unwind": "$sp_info"},
    # Paso 3: Agrupar finalmente por Reino (Kingdom), te_class y te_order
    {"$group": {
        "_id": {
            "Kingdom": "$sp_info.Kingdom",
            "te_class": "$_id.te_class",
            "te_order": "$_id.te_order"
        },
        "total_tes": {"$sum": "$conteo_especie"}
    }},
    {"$sort": {"_id.Kingdom": 1, "total_tes": -1}}
]

df_q3_reino = pd.json_normalize(list(db.sequences.aggregate(pipeline_q3_reino)))
display(df_q3_reino)

# 3B. Cantidad de TEs por Clase y Orden entre Phyla (Phylum)
pipeline_q3_phylum = [
    {"$group": {
        "_id": {
            "species_key": "$species_key",
            "te_class": "$te_class",
            "te_order": "$te_order"
        },
        "conteo_especie": {"$sum": 1}
    }},
    {"$lookup": {
        "from": "species",
        "localField": "_id.species_key",
        "foreignField": "species_key",
        "as": "sp_info"
    }},
    {"$unwind": "$sp_info"},
    {"$group": {
        "_id": {
            "Phylum": "$sp_info.Phylum",
            "te_class": "$_id.te_class",
            "te_order": "$_id.te_order"
        },
        "total_tes": {"$sum": "$conteo_especie"}
    }},
    {"$sort": {"total_tes": -1}}
]

df_q3_phylum = pd.json_normalize(list(db.sequences.aggregate(pipeline_q3_phylum)))
display(df_q3_phylum.head(15))

,total_tes,_id.Kingdom,_id.te_class,_id.te_order
0,44477,Animalia,CLASSI,LINE
1,25382,Animalia,CLASSI,LTR
2,21659,Animalia,CLASSII,TIR
3,3091,Animalia,CLASSI,SINE
4,2718,Animalia,CLASSII,MITE
5,2125,Animalia,CLASSI,DIRS
6,1167,Animalia,CLASSII,HELITRON
7,257,Animalia,CLASSI,PLE
8,113,Animalia,CLASSI,RETROPOSON
9,112,Animalia,CLASSII,CRYPTON


,total_tes,_id.Phylum,_id.te_class,_id.te_order
0,84523,Angiosperma,CLASSI,LTR
1,22638,Chordata,CLASSI,LINE
2,18756,Chordata,CLASSI,LTR
3,17265,Angiosperma,CLASSI,LINE
4,16584,Chordata,CLASSII,TIR
5,15318,Arthropoda,CLASSI,LINE
6,11407,Angiosperma,CLASSII,TIR
7,5070,Arthropoda,CLASSI,LTR
8,4040,Ascomycota,CLASSI,LTR
9,3459,Ascomycota,CLASSI,LINE


### Consulta 4
¿Cuáles son las 10 especies con más TEs y las 10 con menos?

In [5]:
def top_especies(orden=-1, limite=10):
    pipeline = [
        {"$group": {"_id": "$species_key", "total_tes": {"$sum": 1}}},
        {"$sort": {"total_tes": orden, "_id": 1}},
        {"$limit": limite},
        {"$lookup": {
            "from": "species",
            "localField": "_id",
            "foreignField": "species_key",
            "as": "info"
        }},
        {"$unwind": "$info"},
        {"$project": {
            "_id": 0,
            "Species": "$info.Species",
            "Kingdom": "$info.Kingdom",
            "Phylum": "$info.Phylum",
            "total_tes": 1
        }}
    ]
    return pd.DataFrame(list(db.sequences.aggregate(pipeline)))

print("Top 10 especies con MÁS elementos transponibles:")
display(top_especies(orden=-1, limite=10))

print("Top 10 especies con MENOS elementos transponibles:")
display(top_especies(orden=1, limite=10))

Top 10 especies con MÁS elementos transponibles:


,total_tes,Species,Kingdom,Phylum
0,1627,Malus domestica,Plantae,Angiosperma
1,1352,Danio rerio,Animalia,Chordata
2,1294,Callithrix jacchus,Animalia,Chordata
3,1173,Homininae,Animalia,Chordata
4,1159,Nicotiana attenuata,Plantae,Angiosperma
5,1148,Colobus angolensis,Animalia,Chordata
6,1129,Chlorocebus sabaeus,Animalia,Chordata
7,1115,Geotrypetes seraphini,Animalia,Chordata
8,1103,Sitophilus oryzae,Animalia,Arthropoda
9,1090,Oryza sativa,Plantae,Angiosperma


Top 10 especies con MENOS elementos transponibles:


,total_tes,Species,Kingdom,Phylum
0,1,Abortiporus biennis,Fungi,Basidiomycota
1,1,Absidia glauca,Fungi,Mucoromycota
2,1,Achaetomium macrosporum,Fungi,Ascomycota
3,1,Acidothrix acidophila,Fungi,Ascomycota
4,1,Acropora acuminata,Animalia,Cnidaria
5,1,Acropora nasuta,Animalia,Cnidaria
6,1,Agrocybe aegerita,Fungi,Basidiomycota
7,1,Alternaria arborescens,Fungi,Ascomycota
8,1,Alternaria carthami,Fungi,Ascomycota
9,1,Alternaria gaisen,Fungi,Ascomycota


## Bloque 2: Longitud y contenido GC (Consultas 5 a 7)

### Consulta 5
¿Cuál es la longitud promedio, mínima y máxima de los TEs por clase, orden y superfamilia?

In [6]:
pipeline_q5 = [
    {"$group": {
        "_id": {
            "te_class": "$te_class",
            "te_order": "$te_order",
            "te_superfamily": "$te_superfamily"
        },
        "longitud_promedio": {"$avg": "$metrics.length"},
        "longitud_minima": {"$min": "$metrics.length"},
        "longitud_maxima": {"$max": "$metrics.length"},
        "total_tes": {"$sum": 1}
    }},
    {"$project": {
        "_id": 0,
        "te_class": "$_id.te_class",
        "te_order": "$_id.te_order",
        "te_superfamily": "$_id.te_superfamily",
        "longitud_promedio": {"$round": ["$longitud_promedio", 2]},
        "longitud_minima": 1,
        "longitud_maxima": 1,
        "total_tes": 1
    }},
    {"$sort": {"te_class": 1, "te_order": 1, "longitud_promedio": -1}}
]

df_q5 = pd.DataFrame(list(db.sequences.aggregate(pipeline_q5)))
display(df_q5.head(20))

,longitud_minima,longitud_maxima,total_tes,te_class,te_order,te_superfamily,longitud_promedio
0,109,6987,2067,CLASSI,DIRS,DIRS,4772.04
1,191,6998,866,CLASSI,DIRS,NGARO,3557.88
2,515,5992,1368,CLASSI,LINE,LOA,4044.69
3,1030,5958,56,CLASSI,LINE,DUALEN,4029.30
4,503,5995,1990,CLASSI,LINE,I,3804.91
5,512,5986,1176,CLASSI,LINE,R1,3686.80
6,500,6000,23821,CLASSI,LINE,L1,3501.49
7,511,5997,1605,CLASSI,LINE,JOCKEY,3390.13
8,507,5976,1314,CLASSI,LINE,RTEX,3310.43
9,532,5940,629,CLASSI,LINE,CRE,3262.99


### Consulta 6
¿Cuál es el porcentaje de GC promedio por orden y por reino?

In [7]:
pipeline_q6 = [
    # Paso 1: Pre-agrupar por especie y orden de TE sumando GC y contando secuencias
    {"$group": {
        "_id": {"species_key": "$species_key", "te_order": "$te_order"},
        "suma_gc": {"$sum": "$metrics.gc_content"},
        "n_seqs": {"$sum": 1}
    }},
    # Paso 2: Cruzar con species para traer el Reino (Kingdom)
    {"$lookup": {
        "from": "species",
        "localField": "_id.species_key",
        "foreignField": "species_key",
        "as": "sp"
    }},
    {"$unwind": "$sp"},
    # Paso 3: Agrupar por Kingdom y te_order y calcular el promedio exacto
    {"$group": {
        "_id": {"Kingdom": "$sp.Kingdom", "te_order": "$_id.te_order"},
        "total_gc": {"$sum": "$suma_gc"},
        "total_seqs": {"$sum": "$n_seqs"}
    }},
    {"$project": {
        "_id": 0,
        "Kingdom": "$_id.Kingdom",
        "te_order": "$_id.te_order",
        "gc_promedio": {"$round": [{"$divide": ["$total_gc", "$total_seqs"]}, 2]},
        "total_seqs": 1
    }},
    {"$sort": {"Kingdom": 1, "gc_promedio": -1}}
]

df_q6 = pd.DataFrame(list(db.sequences.aggregate(pipeline_q6)))
display(df_q6)

,total_seqs,Kingdom,te_order,gc_promedio
0,3091,Animalia,SINE,53.66
1,2125,Animalia,DIRS,52.64
2,113,Animalia,RETROPOSON,50.80
3,25382,Animalia,LTR,47.68
4,112,Animalia,CRYPTON,45.58
5,1167,Animalia,HELITRON,43.83
6,44477,Animalia,LINE,43.40
7,2718,Animalia,MITE,42.56
8,21659,Animalia,TIR,41.85
9,43,Animalia,MAVERICK,39.52


### Consulta 7
¿Qué TEs son los más largos y los más cortos de cada reino? Muestren su nombre, clasificación y especie.

In [8]:
resultados_q7 = []

for reino in ["Animalia", "Plantae", "Fungi"]:
    # Obtenemos las claves de especies de este reino (consulta instantánea en species)
    claves_reino = db.species.distinct("species_key", {"Kingdom": reino})

    for etiqueta, orden in [("Más largo", -1), ("Más corto", 1)]:
        doc = db.sequences.find_one(
            {"species_key": {"$in": claves_reino}},
            {"_id": 0, "sequence_id": 1, "classification_raw": 1, "species_raw": 1, "metrics.length": 1},
            sort=[("metrics.length", orden)]
        )
        if doc:
            resultados_q7.append({
                "Kingdom": reino,
                "Extremo": etiqueta,
                "sequence_id": doc["sequence_id"],
                "clasificacion": doc["classification_raw"],
                "especie": doc["species_raw"],
                "longitud": doc["metrics"]["length"]
            })

df_q7 = pd.DataFrame(resultados_q7)
display(df_q7)

,Kingdom,Extremo,sequence_id,clasificacion,especie,longitud
0,Animalia,Más largo,PDB00076912,CLASSI/LTR/ERV,Aphelocoma californica,18951
1,Animalia,Más corto,AllBul-4.608,CLASSI/SINE/ID,Orientallactaga bullata,81
2,Plantae,Más largo,PDB00038441,CLASSI/LTR/GYPSY,Helianthus annuus,19926
3,Plantae,Más corto,PDB00049072,CLASSII/HELITRON/HELITRON,Oryza barthii,100
4,Fungi,Más largo,PDB00194372,CLASSI/LTR/ERV,Lactarius subdulcis,19189
5,Fungi,Más corto,PDB00183015,CLASSII/HELITRON/HELITRON,Dactylonectria macrodidyma,100


## Bloque 3: Búsqueda de secuencias y generación de bibliotecas .fasta (Consultas 8 a 11)

### Consulta 8
Buscar una secuencia por su nombre (por ejemplo, PDB00000001) y mostrar su clasificación, especie y taxonomía completa.

In [9]:
def buscar_secuencia_por_id(seq_id="PDB00000001"):
    pipeline = [
        {"$match": {"sequence_id": seq_id}},
        {"$lookup": {
            "from": "species",
            "localField": "species_key",
            "foreignField": "species_key",
            "as": "taxonomia"
        }},
        {"$unwind": "$taxonomia"},
        {"$project": {
            "_id": 0,
            "sequence_id": 1,
            "te_class": 1,
            "te_order": 1,
            "te_superfamily": 1,
            "Species": "$taxonomia.Species",
            "Kingdom": "$taxonomia.Kingdom",
            "Phylum": "$taxonomia.Phylum",
            "Class": "$taxonomia.Class",
            "Order": "$taxonomia.Order",
            "Family": "$taxonomia.Family",
            "length": "$metrics.length",
            "gc_content": "$metrics.gc_content"
        }}
    ]
    return pd.DataFrame(list(db.sequences.aggregate(pipeline)))

display(buscar_secuencia_por_id("PDB00000001"))

,sequence_id,te_class,te_order,te_superfamily,Species,Kingdom,Phylum,Class,Order,Family,length,gc_content
0,PDB00000001,CLASSII,HELITRON,HELITRON,Halicephalobus mephisto,Animalia,Nematoda,Chromadorea,Rhabditida,Panagrolaimidae,1991,31.541939


### Consulta 9
Listar los nombres de todas las secuencias de una superfamilia en una especie dada.

In [10]:
from pipeline.limpiar import clave

def listar_secuencias_superfamilia(especie="Halicephalobus mephisto", superfamilia="HELITRON"):
    sp_key = clave(especie)
    cursor = db.sequences.find(
        {"species_key": sp_key, "te_superfamily": superfamilia.upper()},
        {"_id": 0, "sequence_id": 1, "te_class": 1, "te_order": 1, "te_superfamily": 1, "metrics.length": 1}
    )
    return pd.DataFrame(list(cursor))

df_q9 = listar_secuencias_superfamilia("Halicephalobus mephisto", "HELITRON")
print(f"Total encontradas: {len(df_q9)}")
display(df_q9.head(15))

Total encontradas: 2


,sequence_id,te_class,te_order,te_superfamily,metrics
0,PDB00000001,CLASSII,HELITRON,HELITRON,{'length': 1991}
1,PDB00000002,CLASSII,HELITRON,HELITRON,{'length': 1689}


### Consulta 10
Exportar una biblioteca `.fasta` de una especie dada, directamente desde MongoDB, con el encabezado `>ID#CLASE/ORDEN/SUPERFAMILIA @Especie` y usando parámetros.

In [11]:
from pathlib import Path

CARPETA_FASTA = Path("../exports_fasta")
CARPETA_FASTA.mkdir(exist_ok=True)

# CONSULTA 10: Exportar biblioteca .fasta de una especie dada (parametrizada)
def exportar_biblioteca_especie(nombre_especie, ruta_salida=None):
    sp_key = clave(nombre_especie)
    if ruta_salida is None:
        nombre_limpio = sp_key.replace(" ", "_")
        ruta_salida = CARPETA_FASTA / f"biblioteca_especie_{nombre_limpio}.fasta"

    cursor = db.sequences.find(
        {"species_key": sp_key},
        {"_id": 0, "sequence_id": 1, "classification_raw": 1, "species_raw": 1, "sequence": 1}
    )

    conteo = 0
    with open(ruta_salida, "w", encoding="utf-8") as f:
        for doc in cursor:
            header = f">{doc['sequence_id']}#{doc['classification_raw']} @{doc['species_raw']}\n"
            f.write(header)
            f.write(f"{doc['sequence']}\n")
            conteo += 1

    print(f"[Consulta 10] Exportadas {conteo} secuencias de '{nombre_especie}' en: {ruta_salida}")
    return ruta_salida

### Consulta 11
Exportar una biblioteca `.fasta` de un reino dado, directamente desde MongoDB, con el mismo formato de encabezado y usando parámetros.

In [12]:
# CONSULTA 11: Exportar biblioteca .fasta de un reino dado (parametrizada)
def exportar_biblioteca_reino(nombre_reino, limite=None, ruta_salida=None):
    if ruta_salida is None:
        ruta_salida = CARPETA_FASTA / f"biblioteca_reino_{nombre_reino.lower()}.fasta"

    # 1. Obtener todas las species_key que pertenecen al reino solicitado
    claves_reino = db.species.distinct("species_key", {"Kingdom": nombre_reino})

    # 2. Consultar secuencias filtrando por esas especies
    cursor = db.sequences.find(
        {"species_key": {"$in": claves_reino}},
        {"_id": 0, "sequence_id": 1, "classification_raw": 1, "species_raw": 1, "sequence": 1}
    )
    if limite:
        cursor = cursor.limit(limite)

    conteo = 0
    with open(ruta_salida, "w", encoding="utf-8") as f:
        for doc in cursor:
            header = f">{doc['sequence_id']}#{doc['classification_raw']} @{doc['species_raw']}\n"
            f.write(header)
            f.write(f"{doc['sequence']}\n")
            conteo += 1

    print(f"[Consulta 11] Exportadas {conteo} secuencias del reino '{nombre_reino}' en: {ruta_salida}")
    return ruta_salida

# Ejecución para generar los archivos obligatorios del Entregable 6:
exportar_biblioteca_especie("Halicephalobus mephisto")
exportar_biblioteca_reino("Plantae")  # Plantae tiene 451 especies, genera un archivo manejable

[Consulta 10] Exportadas 10 secuencias de 'Halicephalobus mephisto' en: ..\exports_fasta\biblioteca_especie_halicephalobus_mephisto.fasta
[Consulta 11] Exportadas 119005 secuencias del reino 'Plantae' en: ..\exports_fasta\biblioteca_reino_plantae.fasta


WindowsPath('../exports_fasta/biblioteca_reino_plantae.fasta')

## Bloque 4: Relaciones, calidad, k-mers y consultas propias (Consultas 12 a 17)

### Consulta 12
¿Cómo se puede unir explícitamente una secuencia con la taxonomía completa de su especie utilizando el operador `$lookup`?

In [13]:
pipeline_q12 = [
    {"$limit": 5},  # Muestra de 5 documentos para verificar
    {"$lookup": {
        "from": "species",
        "localField": "species_key",
        "foreignField": "species_key",
        "as": "taxonomia_especie"
    }},
    {"$unwind": "$taxonomia_especie"},
    {"$project": {
        "_id": 0,
        "sequence_id": 1,
        "te_class": 1,
        "especie": "$species_raw",
        "reino": "$taxonomia_especie.Kingdom",
        "phylum": "$taxonomia_especie.Phylum"
    }}
]
df_q12 = pd.DataFrame(list(db.sequences.aggregate(pipeline_q12)))
display(df_q12)

,sequence_id,te_class,especie,reino,phylum
0,AcoCah-1.102,CLASSI,Acomys cahirinus,Animalia,Chordata
1,AcoCah-1.121,CLASSI,Acomys cahirinus,Animalia,Chordata
2,AcoCah-1.12,CLASSI,Acomys cahirinus,Animalia,Chordata
3,AcoCah-1.13,CLASSI,Acomys cahirinus,Animalia,Chordata
4,AcoCah-1.14,CLASSI,Acomys cahirinus,Animalia,Chordata


### Consulta 13
¿Cómo se comparan las especies y secuencias agrupadas según su fuente de origen (Dfam frente a RM2+MCH), evaluando conteos, promedio de secuencias por especie y longitud promedio?

In [14]:
pipeline_q13 = [
    {"$lookup": {
        "from": "species",
        "localField": "species_key",
        "foreignField": "species_key",
        "as": "sp"
    }},
    {"$unwind": "$sp"},
    {"$group": {
        "_id": "$sp.Source",
        "total_secuencias": {"$sum": 1},
        "especies_unicas": {"$addToSet": "$species_key"},
        "longitud_promedio": {"$avg": "$metrics.length"}
    }},
    {"$project": {
        "_id": 0,
        "source": "$_id",
        "total_secuencias": 1,
        "total_especies": {"$size": "$especies_unicas"},
        "promedio_secuencias_por_especie": {"$divide": ["$total_secuencias", {"$size": "$especies_unicas"}]},
        "longitud_promedio": {"$round": ["$longitud_promedio", 2]}
    }}
]
df_q13 = pd.DataFrame(list(db.sequences.aggregate(pipeline_q13)))
display(df_q13)

,total_secuencias,source,total_especies,promedio_secuencias_por_especie,longitud_promedio
0,31592,Dfam,386,81.844560,1653.80
1,208354,RM2+MCH,2404,86.669717,4375.51


### Consulta 14
¿Qué control de calidad podemos aplicar para identificar secuencias que presenten un porcentaje elevado de bases indeterminadas ($N > 5\%$) o anomalías estructurales?

In [15]:
# n_pct se calcula a partir de n_count y length
pipeline_q14 = [
    {"$addFields": {"n_pct_calc": {"$multiply": [{"$divide": ["$metrics.n_count", "$metrics.length"]}, 100]}}},
    {"$match": {"n_pct_calc": {"$gt": 5.0}}},
    {"$project": {
        "_id": 0,
        "sequence_id": 1,
        "species_raw": 1,
        "classification_raw": 1,
        "length": "$metrics.length",
        "n_count": "$metrics.n_count",
        "n_pct": {"$round": ["$n_pct_calc", 2]}
    }},
    {"$sort": {"n_pct": -1}},
    {"$limit": 10}
]
df_q14 = pd.DataFrame(list(db.sequences.aggregate(pipeline_q14)))

total_q14 = db.sequences.count_documents(
    {"$expr": {"$gt": [{"$multiply": [{"$divide": ["$metrics.n_count", "$metrics.length"]}, 100]}, 5.0]}}
)
print(f"Total de secuencias con más del 5% de N: {total_q14}")
display(df_q14)

Total de secuencias con más del 5% de N: 4374


,sequence_id,species_raw,classification_raw,length,n_count,n_pct
0,Halyomorpha_halys-5_family-957,Halyomorpha halys,CLASSII/TIR/TC1MARINER,797,399,50.06
1,TC1DR1,Danio,CLASSII/TIR/TC1MARINER,505,230,45.54
2,Halyomorpha_halys-5_family-2805,Halyomorpha halys,CLASSII/TIR/HAT,395,135,34.18
3,BEL33_DR,Danio rerio,CLASSI/LTR/BELPAO,7203,2310,32.07
4,fAlbLTRK1b,Ficedula albicollis,CLASSI/LTR/ERV,3200,1007,31.47
5,TguLTR13c_fAlb,Ficedula albicollis,CLASSI/LTR/ERV,1591,482,30.30
6,TguLTRL1a7-b_fAlb,Ficedula albicollis,CLASSI/LTR/ERV,1636,482,29.46
7,Penelope-2_DR,Danio rerio,CLASSI/PLE/PLE,3728,1071,28.73
8,hAT-N36B_DR,Danio rerio,CLASSII/TIR/HAT,1383,368,26.61
9,Halyomorpha_halys-6_family-897,Halyomorpha halys,CLASSII/TIR/TC1MARINER,350,88,25.14


### Consulta 15
¿Cómo se puede recuperar el perfil de k-mers almacenado en MongoDB para una secuencia específica y un valor de $k$ dado (por ejemplo, $k=2$)?

In [16]:
def obtener_kmers_secuencia(seq_id="PDB00000001", k=2):
    doc = db.sequences.find_one(
        {"sequence_id": seq_id},
        {f"kmers.k{k}": 1, "_id": 0, "sequence_id": 1}
    )
    return doc

print(obtener_kmers_secuencia("PDB00000001", k=2))

{'sequence_id': 'PDB00000001', 'kmers': {'k2': {'AA': 306, 'AC': 74, 'AG': 110, 'AT': 229, 'CA': 97, 'CC': 45, 'CG': 53, 'CT': 102, 'GA': 123, 'GC': 75, 'GG': 55, 'GT': 78, 'TA': 193, 'TC': 103, 'TG': 113, 'TT': 234}}}


### Consulta 16 (pregunta propia 1)
¿Cuáles son las 5 familias taxonómicas de plantas (Plantae) con mayor cantidad de elementos transponibles registrados?

In [17]:
pipeline_q16 = [
    {"$lookup": {
        "from": "species",
        "localField": "species_key",
        "foreignField": "species_key",
        "as": "sp"
    }},
    {"$unwind": "$sp"},
    {"$match": {"sp.Kingdom": "Plantae"}},
    {"$group": {
        "_id": "$sp.Family",
        "total_tes": {"$sum": 1}
    }},
    {"$sort": {"total_tes": -1}},
    {"$limit": 5}
]
display(pd.DataFrame(list(db.sequences.aggregate(pipeline_q16))))

,_id,total_tes
0,Poaceae,16181
1,Myrtaceae,11306
2,Solanaceae,9181
3,Rosaceae,8250
4,Brassicaceae,6516


### Consulta 17 (pregunta propia 2)
¿Cuál es el contenido de GC promedio en las secuencias del orden de transposones HELITRON en comparación con LTR?

In [18]:
pipeline_q17 = [
    {"$match": {"te_order": {"$in": ["HELITRON", "LTR"]}}},
    {"$group": {
        "_id": "$te_order",
        "gc_promedio": {"$avg": "$metrics.gc_content"},
        "total_tes": {"$sum": 1}
    }},
    {"$project": {
        "_id": 0,
        "te_order": "$_id",
        "gc_promedio": {"$round": ["$gc_promedio", 2]},
        "total_tes": 1
    }}
]
display(pd.DataFrame(list(db.sequences.aggregate(pipeline_q17))))

,total_tes,te_order,gc_promedio
0,119368,LTR,42.91
1,5075,HELITRON,41.28
